## llama.cpp GGUF backend — Colab T4 1x

Fill parameters and run.

In [ ]:

# Notebook bootstrap panel: compact realtime output for clone/install before gguf_backend is available.
import os, sys, time, html, uuid, subprocess, select
from pathlib import Path
import ipywidgets as widgets
from IPython.display import display, Javascript

BOOT_TERM_ID = "boot_term_" + uuid.uuid4().hex
_boot_title = widgets.HTML('<div style="font-family:ui-monospace,monospace;font-weight:700;padding:8px 10px;border:1px solid #333;border-bottom:0;background:#111;color:#f5f5f5;">Bootstrap</div>')
_boot_status = widgets.HTML('<div style="font-family:ui-monospace,monospace;padding:8px 10px;border-left:1px solid #333;border-right:1px solid #333;background:#181818;color:#ddd;">idle</div>')
_boot_term = widgets.HTML(f'<div id="{BOOT_TERM_ID}" style="margin:0;padding:10px;height:260px;overflow-y:auto;white-space:pre-wrap;word-break:break-word;background:#050505;color:#e8e8e8;border:1px solid #333;font-family:ui-monospace,monospace;font-size:13px;line-height:1.35;">ready</div>')
_boot_footer = widgets.HTML('<div style="font-family:ui-monospace,monospace;font-size:12px;padding:7px 10px;border:1px solid #333;border-top:0;background:#111;color:#aaa;">log: -</div>')
display(widgets.VBox([_boot_title, _boot_status, _boot_term, _boot_footer]))
display(Javascript(f"""
(function() {{
  const id = "{BOOT_TERM_ID}";
  const key = "__autoscr_" + id;
  if (window[key]) clearInterval(window[key]);
  window[key] = setInterval(function() {{
    const el = document.getElementById(id);
    if (el) el.scrollTop = el.scrollHeight;
  }}, 100);
}})();
"""))

def _esc(x):
    return html.escape(str(x), quote=False)

def _set_boot_status(text):
    _boot_status.value = f'<div style="font-family:ui-monospace,monospace;padding:8px 10px;border-left:1px solid #333;border-right:1px solid #333;background:#181818;color:#ddd;">{_esc(text)}</div>'

def _set_boot_term(lines):
    body = "\n".join(lines[-300:])
    _boot_term.value = f'<div id="{BOOT_TERM_ID}" style="margin:0;padding:10px;height:260px;overflow-y:auto;white-space:pre-wrap;word-break:break-word;background:#050505;color:#e8e8e8;border:1px solid #333;font-family:ui-monospace,monospace;font-size:13px;line-height:1.35;">{_esc(body)}</div>'

def bootstrap_run(cmd, *, label="bootstrap", cwd=None, check=True):
    log_dir = Path("/content/_logs" if Path("/kaggle/working").exists() else "/content/_logs")
    log_dir.mkdir(parents=True, exist_ok=True)
    log_path = log_dir / (label.replace(" ", "_") + ".log")
    _boot_footer.value = f'<div style="font-family:ui-monospace,monospace;font-size:12px;padding:7px 10px;border:1px solid #333;border-top:0;background:#111;color:#aaa;">log: {_esc(log_path)}</div>'
    lines = ["$ " + (cmd if isinstance(cmd, str) else " ".join(map(str, cmd)))]
    _set_boot_term(lines)
    _set_boot_status(f"running: {label}")
    p = subprocess.Popen(cmd, shell=isinstance(cmd, str), cwd=cwd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, stdin=subprocess.DEVNULL, bufsize=0)
    fd = p.stdout.fileno()
    os.set_blocking(fd, False)
    start = time.time(); last = 0
    with open(log_path, "w", encoding="utf-8", errors="replace") as f:
        while True:
            ready, _, _ = select.select([fd], [], [], 0.1)
            if ready:
                raw = os.read(fd, 8192)
                if raw:
                    text = raw.decode("utf-8", errors="replace")
                    f.write(text); f.flush()
                    for line in text.replace("\r", "\n").splitlines():
                        line = line.strip()
                        if line:
                            if len(line) > 280: line = line[:280] + " ..."
                            lines.append(line)
            if time.time() - last > 0.15:
                _set_boot_status(f"running: {label} | elapsed {int(time.time()-start)}s")
                _set_boot_term(lines)
                last = time.time()
            if p.poll() is not None:
                break
    rc = p.wait()
    lines.append(f"$ exit {rc}")
    _set_boot_term(lines)
    _set_boot_status(f"completed: {label}" if rc == 0 else f"failed: {label} | exit {rc}")
    if check and rc != 0:
        raise RuntimeError(f"{label} failed, check log: {log_path}")
    return rc

#@title llama.cpp GGUF backend
REPO_URL = "https://github.com/N3iKos/llama-cpp-notebook" #@param {type:"string"}
REPO_BRANCH = "main" #@param {type:"string"}
MODEL_URL = "https://huggingface.co/ggml-org/Qwen2.5-VL-3B-Instruct-GGUF/resolve/main/Qwen2.5-VL-3B-Instruct-Q4_K_M.gguf" #@param {type:"string"}
MMPROJ_URL = "https://huggingface.co/ggml-org/Qwen2.5-VL-3B-Instruct-GGUF/resolve/main/mmproj-Qwen2.5-VL-3B-Instruct-Q8_0.gguf" #@param {type:"string"}
HF_TOKEN = "" #@param {type:"string"}
NGROK_AUTHTOKEN = "" #@param {type:"string"}
TUNNEL_MODE = "both" #@param ["both", "ngrok", "cloudflare"]
CTX_SIZE = 8192 #@param {type:"integer"}
BATCH_SIZE = 2048 #@param {type:"integer"}
UBATCH_SIZE = 512 #@param {type:"integer"}
PARALLEL = 1 #@param {type:"integer"}
ENABLE_FLASH_ATTN = True #@param {type:"boolean"}
ENABLE_MMPROJ_OFFLOAD = True #@param {type:"boolean"}
IMAGE_MIN_TOKENS = "" #@param {type:"string"}
IMAGE_MAX_TOKENS = "" #@param {type:"string"}
CHAT_TEMPLATE_KWARGS = "" #@param {type:"string"}

repo_dir = Path("/content/llama-cpp-notebook")
if not repo_dir.exists():
    bootstrap_run(["git", "clone", "--depth", "1", "--branch", REPO_BRANCH, REPO_URL, str(repo_dir)], label="clone repo")
else:
    bootstrap_run(["git", "-C", str(repo_dir), "pull"], label="update repo", check=False)

bootstrap_run([sys.executable, "-m", "pip", "install", "-q", "-e", str(repo_dir)], label="install package")
sys.path.insert(0, str(repo_dir))

def optional_int(value):
    value = str(value).strip()
    return int(value) if value else None

from gguf_backend.colab_runner import run_colab
from gguf_backend.panel import show_summary

result = run_colab(
    model_url=MODEL_URL,
    mmproj_url=MMPROJ_URL,
    hf_token=HF_TOKEN,
    ngrok_authtoken=NGROK_AUTHTOKEN,
    tunnel_mode=TUNNEL_MODE,
    ctx_size=CTX_SIZE,
    split_mode="none",
    tensor_split="1",
    batch_size=BATCH_SIZE,
    ubatch_size=UBATCH_SIZE,
    parallel=PARALLEL,
    flash_attn=ENABLE_FLASH_ATTN,
    image_min_tokens=optional_int(IMAGE_MIN_TOKENS),
    image_max_tokens=optional_int(IMAGE_MAX_TOKENS),
    chat_template_kwargs=CHAT_TEMPLATE_KWARGS.strip() or None,
    mmproj_offload=ENABLE_MMPROJ_OFFLOAD,
    port=8080,
    alias="local-vl",
)

show_summary("colab run result", result)
